# EDA — Dataset Diabetes 130-US Hospitals (1999–2008)
**MLOps Proyecto 2 — Pontificia Universidad Javeriana**

Este notebook presenta el análisis exploratorio del dataset de diabetes clínica y justifica las decisiones de preprocesamiento implementadas en `training/preprocessing.py`.

**Dataset:** 10 años de datos clínicos de 130 hospitales de EE.UU., ~101,766 registros y 50 características.

**Objetivo:** Predecir si un paciente diabético será readmitido en menos de 30 días (`readmitted = <30` → clase positiva).

## 1. Carga y exploración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

# Ajustar la ruta según tu entorno
DATA_PATH = '../diabetic_data.csv'

df = pd.read_csv(DATA_PATH)
df = df.replace('?', np.nan)  # '?' representa valores faltantes en este dataset

print(f'Shape: {df.shape}')
print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')

In [ ]:
df.dtypes.value_counts()

In [ ]:
df.head(3)

In [ ]:
df.describe(include='all').T[['count','unique','top','freq','mean','std','min','max']].head(30)

## 2. Análisis de valores faltantes

El dataset usa `'?'` para representar valores ausentes, que ya fueron reemplazados por `NaN`.

In [ ]:
missing = df.isnull().mean().sort_values(ascending=False)
missing_pct = (missing * 100).round(2)
missing_df = pd.DataFrame({'missing_pct': missing_pct, 'missing_count': df.isnull().sum()})
print('Columnas con valores faltantes:')
print(missing_df[missing_df.missing_pct > 0].to_string())

In [ ]:
cols_with_missing = missing_df[missing_df.missing_pct > 0]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cols_with_missing.index, cols_with_missing.missing_pct, color='steelblue')
ax.axvline(50, color='red', linestyle='--', label='Umbral 50%')
ax.set_xlabel('% Valores faltantes')
ax.set_title('Porcentaje de valores faltantes por columna')
ax.legend()
plt.tight_layout()
plt.show()

### Justificación de columnas eliminadas

| Columna | Razón de eliminación |
|---|---|
| `weight` | >96% valores faltantes — imputar sería introducir ruido extremo |
| `payer_code` | ~40% faltantes; variable administrativa sin valor predictivo clínico |
| `medical_specialty` | ~49% faltantes; alta cardinalidad (70+ valores únicos) |
| `diag_1`, `diag_2`, `diag_3` | Códigos ICD-9 con >900 categorías únicas — requieren ontología médica especializada |
| `encounter_id` | Identificador técnico, no aporta información predictiva |
| `patient_nbr` | Identificador de paciente, puede generar data leakage |

**Criterio general:** se eliminan columnas con >50% de nulos o con cardinalidad excesiva que no puede codificarse de forma determinista.

## 3. Variable objetivo: `readmitted`

In [ ]:
target_counts = df['readmitted'].value_counts()
print(target_counts)
print(f'\nDistribución (%):')
print((target_counts / len(df) * 100).round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribución original
target_counts.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#2196F3', '#F44336'], edgecolor='black')
axes[0].set_title('Distribución original de readmitted')
axes[0].set_xlabel('Categoría')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(axis='x', rotation=0)

# Tras binarización
binary_map = {'<30': 1, '>30': 0, 'NO': 0}
df['readmitted_binary'] = df['readmitted'].map(binary_map)
binary_counts = df['readmitted_binary'].value_counts()
binary_counts.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#F44336'], edgecolor='black')
axes[1].set_title('Readmitted binarizado (<30 días = 1)')
axes[1].set_xticklabels(['No readmitido (0)', 'Readmitido <30d (1)'], rotation=0)
axes[1].set_ylabel('Cantidad')

plt.tight_layout()
plt.show()

pos_rate = binary_counts[1] / len(df) * 100
print(f'\nDesbalance de clases: {pos_rate:.1f}% positivos (readmitidos <30d)')
print('→ Dataset desbalanceado. Se justifica usar F1-score y ROC-AUC como métricas principales.')

### Justificación de la estrategia de binarización

La variable `readmitted` tiene 3 valores: `<30`, `>30`, `NO`.

Se binariza como: **`<30` → 1 (positivo), `>30` y `NO` → 0 (negativo)**.

**Razón:** El objetivo clínico principal es identificar pacientes en riesgo de **readmisión temprana (<30 días)**, que es el indicador hospitalario de mayor impacto y el más costoso. Las readmisiones tardías (>30 días) se tratan igual que no readmitidos porque no son atribuibles directamente al episodio hospitalario.

**Métricas elegidas:** dado el fuerte desbalance (~11% positivos), se usa **F1-score** y **ROC-AUC** en lugar de accuracy, que sería engañosa.

## 4. Distribución de variables numéricas

In [ ]:
num_cols = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses'
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
    axes[i].set_xlabel('Valor')
    axes[i].set_ylabel('Frecuencia')

plt.suptitle('Distribución de variables numéricas', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
df[num_cols].describe().round(2)

In [ ]:
# Relación con el target binario
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    df.boxplot(column=col, by='readmitted_binary', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Readmitido <30d')
    axes[i].set_ylabel('Valor')

plt.suptitle('Variables numéricas vs. readmitted (0=no, 1=sí)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Variables categóricas clave

In [ ]:
# Distribución de edad
age_order = ['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)',
             '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['age'].value_counts().reindex(age_order).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución de edad')
axes[0].set_xlabel('Grupo etario')
axes[0].tick_params(axis='x', rotation=45)

df['race'].value_counts().head(6).plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Distribución de raza')
axes[1].tick_params(axis='x', rotation=45)

df['gender'].value_counts().plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Distribución de género')
axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Medicamentos: metformina e insulina
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, med in zip(axes, ['metformin', 'insulin']):
    df[med].value_counts().plot(kind='bar', ax=ax, edgecolor='black')
    ax.set_title(f'Distribución de {med}')
    ax.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# A1C result
a1c_col = 'A1Cresult' if 'A1Cresult' in df.columns else 'a1cresult'
df[a1c_col].value_counts().plot(kind='bar', color='purple', edgecolor='black')
plt.title('Resultado de HbA1c')
plt.xlabel('Resultado')
plt.ylabel('Frecuencia')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Estrategia de codificación de variables categóricas

Se usa **codificación ordinal determinista con mapeos fijos** en lugar de one-hot encoding o label encoding basado en datos. Esto garantiza que el pipeline sea **reproducible e idempotente** sin necesidad de fitear un encoder.

| Variable | Estrategia | Justificación |
|---|---|---|
| `age` | Punto medio del rango | Relación ordinal clara; el punto medio preserva la magnitud |
| `race` | Mapeo numérico fijo (1-5) | Categorías bien definidas; `Other/Unknown` → 5 |
| `gender` | Binario (0/1) | Variable nominal con 2 valores relevantes |
| `admission_type_id` | Mapeo a 3 categorías clínicas | Emergency=1, Urgent=2, Elective=3, Otros=0 |
| `discharge_disposition_id` | Mapeo a 4 categorías | Home=1, Hosp=2, SNF=3, Expired=4, Otros=0 |
| `admission_source_id` | Mapeo a 2 categorías | Physician=1, Emergency Room=2, Otros=0 |
| `A1Cresult` | Ordinal (0-3) | Norm=1, >7=2, >8=3, None=0 — orden clínico de gravedad |
| `metformin`, `insulin` | Ordinal (0-3) | No=0, Steady=1, Up=2, Down=3 — intensidad del tratamiento |

**Ventaja sobre OHE:** evita la explosión dimensional y el problema de columnas desconocidas en producción.

## 7. Análisis de correlación

In [ ]:
# Aplicar encoding para análisis de correlación
AGE_MAP = {'[0-10)': 5, '[10-20)': 15, '[20-30)': 25, '[30-40)': 35, '[40-50)': 45,
           '[50-60)': 55, '[60-70)': 65, '[70-80)': 75, '[80-90)': 85, '[90-100)': 95}
RACE_MAP = {'Caucasian': 1, 'AfricanAmerican': 2, 'Hispanic': 3, 'Asian': 4, 'Other': 5}
GENDER_MAP = {'Male': 0, 'Female': 1}
MED_MAP = {'No': 0, 'Steady': 1, 'Up': 2, 'Down': 3}
A1C_MAP = {'Norm': 1, '>7': 2, '>8': 3, 'None': 0}

df_enc = df.copy()
df_enc['age_enc'] = df_enc['age'].map(AGE_MAP).fillna(45)
df_enc['race_enc'] = df_enc['race'].map(RACE_MAP).fillna(5)
df_enc['gender_enc'] = df_enc['gender'].map(GENDER_MAP).fillna(0)
df_enc['metformin_enc'] = df_enc['metformin'].map(MED_MAP).fillna(0)
df_enc['insulin_enc'] = df_enc['insulin'].map(MED_MAP).fillna(0)
a1c_col = 'A1Cresult' if 'A1Cresult' in df_enc.columns else 'a1cresult'
df_enc['a1c_enc'] = df_enc[a1c_col].map(A1C_MAP).fillna(0)

corr_cols = num_cols + ['age_enc', 'race_enc', 'gender_enc', 'metformin_enc', 'insulin_enc', 'a1c_enc', 'readmitted_binary']
corr_matrix = df_enc[corr_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Matriz de correlación — Features + Target', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación con el target
target_corr = corr_matrix['readmitted_binary'].drop('readmitted_binary').sort_values(key=abs, ascending=False)
print('Correlación de cada feature con readmitted_binary:')
print(target_corr.round(4).to_string())

## 8. Duplicados y calidad general

In [ ]:
n_duplicates = df.duplicated().sum()
print(f'Filas duplicadas: {n_duplicates} ({n_duplicates/len(df)*100:.2f}%)')

# Pacientes con múltiples encuentros
if 'patient_nbr' in df.columns:
    multi_encounter = df['patient_nbr'].value_counts()
    print(f'\nPacientes únicos: {df["patient_nbr"].nunique():,}')
    print(f'Encuentros por paciente (promedio): {multi_encounter.mean():.2f}')
    print(f'Pacientes con >1 encuentro: {(multi_encounter > 1).sum():,}')
    print('→ Se elimina patient_nbr para evitar data leakage entre train/test.')

## 9. Resumen de decisiones de preprocesamiento

### Features finales del modelo (contrato Persona 2 ↔ Persona 3)

```
Numéricas (sin transformación):
  time_in_hospital, num_lab_procedures, num_procedures, num_medications,
  number_outpatient, number_emergency, number_inpatient, number_diagnoses

Codificadas con mapeos deterministas:
  age              → punto medio del rango etario
  race_encoded     → categorías clínicas (1-5)
  gender_encoded   → binario (0/1)
  admission_type_encoded   → categorías de urgencia (0-3)
  discharge_encoded        → destino al alta (0-4)
  admission_source_encoded → fuente de ingreso (0-2)
  a1c_result_encoded       → resultado HbA1c (0-3)
  metformin_encoded        → intensidad de tratamiento (0-3)
  insulin_encoded          → intensidad de tratamiento (0-3)

Target: readmitted → 1 si <30 días, 0 en caso contrario
```

### Estrategias de imputación
- **Variables numéricas:** mediana (robusta a outliers)
- **Variables categóricas:** moda
- No se usan métricas aprendidas de datos para evitar data leakage en el pipeline de producción

### Métricas de evaluación justificadas
- **F1-Score (clase positiva):** dado el desbalance (~11% positivos), accuracy es engañosa
- **ROC-AUC:** mide la capacidad discriminativa general del modelo
- **Recall:** importante en contexto clínico para minimizar falsos negativos
- **Precision:** relevante para evitar intervenciones innecesarias

In [ ]:
# Verificación final: el preprocessing.py produce el schema correcto
import sys
sys.path.insert(0, '..')

try:
    from training.preprocessing import run_preprocessing
    df_clean = run_preprocessing(df.copy())
    print(f'Shape tras preprocesamiento: {df_clean.shape}')
    print(f'Columnas: {list(df_clean.columns)}')
    print(f'\nDistribución del target tras binarización:')
    print(df_clean['readmitted'].value_counts())
    print(f'\nTipos de datos:')
    print(df_clean.dtypes.to_string())
except ImportError:
    print('Para ejecutar run_preprocessing, correr este notebook desde la raíz del proyecto.')